In [ ]:
!pip install librosa

In [1]:
import os
import random
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
# ================= PATHS ================= #

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

STEMS_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUPS_PATH = os.path.join(BASE_PATH, "mashups")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
ESC_AUDIO = os.path.join(BASE_PATH, "ESC-50-master/audio")

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]


2026-04-11 06:17:36.574195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775888256.804481      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775888256.871492      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775888257.427305      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775888257.427350      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775888257.427353      55 computation_placer.cc:177] computation placer alr

In [2]:
#test data size:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print("Number of test mashups:", len(test_df))
test_df.head()

Number of test mashups: 3020


,id,filename
0,1,mashups/song0001.wav
1,2,mashups/song0002.wav
2,3,mashups/song0003.wav
3,4,mashups/song0004.wav
4,5,mashups/song0005.wav


In [3]:
!pip install wandb -q

In [1]:

import wandb
import os

from kaggle_secrets import UserSecretsClient

# Get API key from Kaggle secrets
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login()

# Init run
wandb.init(
    project="milestone2",
    name="final-ast-model",
    config={
        "features": "MFCC + Mel + Raw Audio",
        "models": ["LogisticRegression", "NaiveBayes", "CNN", "HuBERT", "AST"],
        "metric": "MacroF1"
    }
)

print("W&B initialized successfully")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

W&B initialized successfully


## EDA

In [2]:
#Class distribution (songs per genre)
genre_counts = {}

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]
    genre_counts[g] = len(songs)

df_genres = pd.DataFrame.from_dict(
    genre_counts, orient="index", columns=["num_songs"]
)

df_genres


,num_songs
blues,100
classical,100
country,100
disco,100
hiphop,100
jazz,100
metal,100
pop,100
reggae,100
rock,100


In [3]:
#Stem availability / gaps
stem_files = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
stem_stats = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in songs:
        song_dir = os.path.join(genre_dir, s)
        available = sum(
            os.path.exists(os.path.join(song_dir, stem))
            for stem in stem_files
        )
        stem_stats.append({
            "genre": g,
            "song": s,
            "num_stems": available
        })

df_stems = pd.DataFrame(stem_stats)
df_stems["num_stems"].value_counts().sort_index()



num_stems
3    1000
Name: count, dtype: int64

In [4]:
#Audio length analysis — training stems
durations = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in random.sample(songs, 5):  # sample few per genre
        song_dir = os.path.join(genre_dir, s)
        for stem in stem_files:
            stem_path = os.path.join(song_dir, stem)
            if os.path.exists(stem_path):
                audio, sr = librosa.load(stem_path, sr=None)
                durations.append(len(audio) / sr)

pd.Series(durations).describe()


count    150.000000
mean      30.020542
std        0.052253
min       30.000181
25%       30.000181
50%       30.013333
75%       30.013333
max       30.291156
dtype: float64

In [5]:
#Mashup duration stats
test_lengths = []

for fname in test_df["filename"].sample(20, random_state=42):
    path = os.path.join(BASE_PATH, fname)
    audio, sr = librosa.load(path, sr=None)
    test_lengths.append(len(audio) / sr)

pd.Series(test_lengths).describe()


NameError: name 'test_df' is not defined

In [ ]:
#esc 50
noise_file = random.choice(os.listdir(ESC_AUDIO))
noise_path = os.path.join(ESC_AUDIO, noise_file)

noise, sr = librosa.load(noise_path, sr=22050)
print("Noise file:", noise_file)
print("Duration:", len(noise)/sr)

noise_spec = librosa.feature.melspectrogram(y=noise, sr=sr)
noise_log = librosa.power_to_db(noise_spec)

plt.figure(figsize=(10, 3))
plt.imshow(noise_log, aspect="auto", origin="lower")
plt.title("ESC-50 Noise Spectrogram")
plt.colorbar()
plt.show()


## model building

In [ ]:
!pip install numpy==1.26.4 librosa --quiet

In [3]:


# # ================= CONFIG ================= #

# SR = 16000
# DURATION = 5
# SAMPLES = SR * DURATION
# MIN_STEMS_REQUIRED = 2

# random.seed(42)
# np.random.seed(42)

# # ================= AUDIO ================= #

# def load_audio(path):
#     audio, _ = librosa.load(path, sr=SR, mono=True)
#     if len(audio) < SAMPLES:
#         audio = np.pad(audio, (0, SAMPLES - len(audio)))
#     return audio[:SAMPLES]

# def extract_mel(audio):
#     mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=64)
#     mel_db = librosa.power_to_db(mel, ref=np.max)
#     return mel_db.astype(np.float32)

# # ================= BUILD DATA ================= #

# X_train, y_train = [], []

# print("Building dataset...")

# for genre in GENRES:
#     genre_path = os.path.join(STEMS_PATH, genre)

#     song_dirs = [
#         d for d in os.listdir(genre_path)
#         if os.path.isdir(os.path.join(genre_path, d))
#     ]

#     for _ in range(100):
#         chosen = random.sample(song_dirs, 4)
#         mix = np.zeros(SAMPLES)
#         used = 0

#         for stem_name, song in zip(
#             ["drums.wav", "bass.wav", "vocals.wav", "others.wav"],
#             chosen
#         ):
#             stem_path = os.path.join(genre_path, song, stem_name)
#             if not os.path.exists(stem_path):
#                 continue

#             mix += load_audio(stem_path)
#             used += 1

#         if used < MIN_STEMS_REQUIRED:
#             continue

#         mix = mix / (np.max(np.abs(mix)) + 1e-6)

#         X_train.append(extract_mel(mix))
#         y_train.append(genre)

# X_train = np.array(X_train)
# y_train = np.array(y_train)

# print("Total samples:", len(X_train))

# # ================= LABEL ENCODING ================= #

# label_map = {g: i for i, g in enumerate(GENRES)}
# inv_label_map = {i: g for g, i in label_map.items()}

# # ================= DATASET ================= #

# class AudioDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = X
#         self.y = y

#     def __len__(self):
#         return len(self.X)

#     def __getitem__(self, idx):
#         x = self.X[idx]
#         x = np.expand_dims(x, axis=0)  # (1, H, W)
#         y = label_map[self.y[idx]]
#         return torch.tensor(x), torch.tensor(y)

# # ================= SPLIT ================= #

# X_tr, X_val, y_tr, y_val = train_test_split(
#     X_train, y_train,
#     test_size=0.2,
#     stratify=y_train,
#     random_state=42
# )

# train_loader = DataLoader(AudioDataset(X_tr, y_tr), batch_size=16, shuffle=True)
# val_loader = DataLoader(AudioDataset(X_val, y_val), batch_size=16)

# # ================= MODEL ================= #

# class SimpleCNN(nn.Module):
#     def __init__(self):
#         super().__init__()

#         self.conv = nn.Sequential(
#             nn.Conv2d(1, 16, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),

#             nn.Conv2d(16, 32, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2)
#         )

#         self.pool = nn.AdaptiveAvgPool2d((4, 4))

#         self.fc = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(32 * 4 * 4, 128),
#             nn.ReLU(),
#             nn.Linear(128, 10)
#         )

#     def forward(self, x):
#         x = self.conv(x)
#         x = self.pool(x)
#         return self.fc(x)

# # ================= TRAIN SETUP ================= #

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = SimpleCNN().to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # ================= TRAIN ================= #

# def train_epoch():
#     model.train()
#     total_loss = 0

#     for x, y in train_loader:
#         x, y = x.to(device), y.to(device)

#         optimizer.zero_grad()
#         out = model(x)
#         loss = criterion(out, y)

#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     return total_loss / len(train_loader)

# # ================= EVAL ================= #

# def evaluate():
#     model.eval()
#     preds, targets = [], []

#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(device)
#             out = model(x)

#             p = torch.argmax(out, dim=1).cpu().numpy()
#             preds.extend(p)
#             targets.extend(y.numpy())

#     return f1_score(targets, preds, average="macro")

# # ================= RUN ================= #

# EPOCHS = 5

# for epoch in range(EPOCHS):
#     loss = train_epoch()
#     f1 = evaluate()

#     wandb.log({"epoch": epoch, "loss": loss, "val_f1": f1})
#     print(f"Epoch {epoch} | Loss: {loss:.4f} | F1: {f1:.4f}")

# wandb.finish()

Building dataset...
Total samples: 1000


NameError: name 'wandb' is not defined

In [2]:
# # ================= TEST INFERENCE ================= #

# print("Running CNN inference on test data...")

# test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

# model.eval()
# predictions = []

# with torch.no_grad():
#     for fname in test_df["filename"]:
#         audio_path = os.path.join(BASE_PATH, fname)

#         audio = load_audio(audio_path)
#         mel = extract_mel(audio)

#         x = np.expand_dims(mel, axis=0)   # (1, H, W)
#         x = np.expand_dims(x, axis=0)     # (1, 1, H, W)

#         x = torch.tensor(x).to(device)

#         out = model(x)
#         pred = torch.argmax(out, dim=1).item()

#         predictions.append(inv_label_map[pred])


# # ================= SUBMISSION ================= #

# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": predictions
# })

# OUT_PATH = "/kaggle/working/submission.csv"
# submission.to_csv(OUT_PATH, index=False)

# print("Submission saved at:", OUT_PATH)
# print(submission.head())

Running CNN inference on test data...
Submission saved at: /kaggle/working/submission.csv
   id   genre
0   1   blues
1   2    rock
2   3  hiphop
3   4   blues
4   5   blues


In [6]:
# # ================= CNN VALIDATION ================= #

# from sklearn.metrics import f1_score

# model.eval()

# all_preds = []
# all_targets = []

# with torch.no_grad():
#     for x, y in val_loader:
#         x = x.to(device)

#         outputs = model(x)
#         preds = torch.argmax(outputs, dim=1).cpu().numpy()

#         all_preds.extend(preds)
#         all_targets.extend(y.numpy())

# cnn_f1 = f1_score(all_targets, all_preds, average="macro")

# print("CNN Validation Macro F1:", cnn_f1)

CNN Validation Macro F1: 0.40696433285210737


In [ ]:
# wandb.finish()

In [ ]:
#milestone 5

In [1]:
# from transformers import HubertModel, Wav2Vec2FeatureExtractor
# import torch.nn as nn
# import torch

2026-03-22 07:59:30.020875: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774166370.179912      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774166370.224484      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774166370.585650      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774166370.585684      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774166370.585686      55 computation_placer.cc:177] computation placer alr

In [9]:
# # ================= INSTALL ================= #
# !pip install transformers -q

# # ================= IMPORTS ================= #
# import os
# import random
# import librosa
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn

# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import f1_score
# from transformers import AutoFeatureExtractor, ASTForAudioClassification

# # ================= PATHS ================= #
# BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# STEMS_PATH = os.path.join(BASE_PATH, "genres_stems")
# TEST_CSV = os.path.join(BASE_PATH, "test.csv")
# ESC_AUDIO = os.path.join(BASE_PATH, "ESC-50-master/audio")

# # ================= CONFIG ================= #
# GENRES = [
#     "blues", "classical", "country", "disco", "hiphop",
#     "jazz", "metal", "pop", "reggae", "rock"
# ]

# SR = 16000
# DURATION = 10
# SAMPLES = SR * DURATION

# # ================= LABEL MAP ================= #
# label_map = {g: i for i, g in enumerate(GENRES)}
# inv_label_map = {i: g for g, i in label_map.items()}

# # ================= AUDIO ================= #
# def load_audio(path):
#     audio, _ = librosa.load(path, sr=SR, mono=True)

#     if len(audio) < SAMPLES:
#         audio = np.pad(audio, (0, SAMPLES - len(audio)))
#     else:
#         audio = audio[:SAMPLES]

#     return audio

# # ================= FEATURE EXTRACTOR ================= #
# feature_extractor = AutoFeatureExtractor.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593"
# )

# # ================= NOISE FILES ================= #
# noise_files = [
#     os.path.join(ESC_AUDIO, f)
#     for f in os.listdir(ESC_AUDIO)
#     if f.endswith(".wav")
# ]

# # ================= DATASET ================= #
# class ASTDataset(Dataset):
#     def __init__(self, audio_list, labels):
#         self.audio = audio_list
#         self.labels = labels

#     def __len__(self):
#         return len(self.audio)

#     def __getitem__(self, idx):
#         y = self.audio[idx]

#         inputs = feature_extractor(
#             y,
#             sampling_rate=SR,
#             return_tensors="pt"
#         )

#         x = inputs["input_values"].squeeze(0)
#         label = label_map[self.labels[idx]]

#         return x, torch.tensor(label)

# # ================= BUILD DATA ================= #
# X_audio, y_audio = [], []

# print("Building AST dataset...")

# for genre in GENRES:
#     genre_path = os.path.join(STEMS_PATH, genre)

#     song_dirs = [
#         d for d in os.listdir(genre_path)
#         if os.path.isdir(os.path.join(genre_path, d))
#     ]

#     for _ in range(100):   # 🔥 optimal
#         chosen = random.sample(song_dirs, 4)
#         mix = np.zeros(SAMPLES)
#         used = 0

#         for stem, song in zip(
#             ["drums.wav", "bass.wav", "vocals.wav", "others.wav"],
#             chosen
#         ):
#             path = os.path.join(genre_path, song, stem)
#             if not os.path.exists(path):
#                 continue

#             mix += load_audio(path)
#             used += 1

#         if used < 2:
#             continue

#         # 🔥 NOISE AUGMENTATION
#         if random.random() < 0.7:
#             noise = load_audio(random.choice(noise_files))
#             mix += 0.2 * noise

#         # normalize
#         mix = mix / (np.max(np.abs(mix)) + 1e-9)

#         X_audio.append(mix.astype(np.float32))
#         y_audio.append(genre)

# print("Total samples:", len(X_audio))

# # ================= SPLIT ================= #
# X_tr, X_val, y_tr, y_val = train_test_split(
#     X_audio, y_audio, test_size=0.2,
#     stratify=y_audio, random_state=42
# )

# train_loader = DataLoader(ASTDataset(X_tr, y_tr), batch_size=8, shuffle=True)
# val_loader = DataLoader(ASTDataset(X_val, y_val), batch_size=8)

# # ================= MODEL ================= #
# model = ASTForAudioClassification.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593",
#     num_labels=10,
#     ignore_mismatched_sizes=True
# )

# # 🔥 FREEZE BACKBONE
# for param in model.base_model.parameters():
#     param.requires_grad = False

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
# criterion = nn.CrossEntropyLoss()

# # ================= TRAIN ================= #
# def train_epoch():
#     model.train()
#     total_loss = 0

#     for x, y in train_loader:
#         x, y = x.to(device), y.to(device)

#         optimizer.zero_grad()
#         out = model(x).logits
#         loss = criterion(out, y)

#         loss.backward()

#         # 🔥 GRAD CLIP
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         optimizer.step()
#         total_loss += loss.item()

#     return total_loss / len(train_loader)

# # ================= EVAL ================= #
# def evaluate():
#     model.eval()
#     preds, targets = [], []

#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(device)

#             out = model(x).logits
#             p = torch.argmax(out, dim=1).cpu().numpy()

#             preds.extend(p)
#             targets.extend(y.numpy())

#     return f1_score(targets, preds, average="macro")

# # ================= TRAIN LOOP ================= #
# EPOCHS = 5

# for epoch in range(EPOCHS):
#     loss = train_epoch()
#     f1 = evaluate()

#     print(f"[AST] Epoch {epoch} | Loss: {loss:.4f} | F1: {f1:.4f}")

# # ================= TEST INFERENCE ================= #
# print("Running AST inference...")

# test_df = pd.read_csv(TEST_CSV)
# predictions = []

# model.eval()

# with torch.no_grad():
#     for fname in test_df["filename"]:
#         path = os.path.join(BASE_PATH, fname)

#         y = load_audio(path)
#         y = y / (np.max(np.abs(y)) + 1e-9)

#         inputs = feature_extractor(
#             y,
#             sampling_rate=SR,
#             return_tensors="pt"
#         )

#         x = inputs["input_values"].to(device)

#         out = model(x).logits
#         pred = torch.argmax(out, dim=1).item()

#         predictions.append(inv_label_map[pred])

# # ================= SUBMISSION ================= #
# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": predictions
# })

# submission.to_csv("submission_ast.csv", index=False)
# print("Saved submission_ast.csv")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Building AST dataset...
Total samples: 1000


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[AST] Epoch 0 | Loss: 2.4206 | F1: 0.1565
[AST] Epoch 1 | Loss: 2.2421 | F1: 0.2005
[AST] Epoch 2 | Loss: 2.0988 | F1: 0.3005


KeyboardInterrupt: 

In [13]:

# print("===== FINAL MODEL COMPARISON =====")

# results = {
#     "Logistic Regression": ,   # e.g. 0.32
#     "Naive Bayes": None,           # e.g. 0.28
#     "CNN": None,                  # e.g. 0.45
#     "HuBERT": None                # e.g. 0.62
# }


# # Print results
# for model, score in results.items():
#     print(f"{model} F1: {score:.4f}")

# # ================= BEST MODEL ================= #

# best_model = max(results, key=results.get)
# best_score = results[best_model]

# print("\n===== BEST MODEL =====")
# print(f"{best_model} achieved highest Macro F1: {best_score:.4f}")

# # ================= W&B LOGGING ================= #

# wandb.log({
#     "LR_F1": results["Logistic Regression"],
#     "NB_F1": results["Naive Bayes"],
#     "CNN_F1": results["CNN"],
#     "HuBERT_F1": results["HuBERT"],
#     "Best_Model_F1": best_score
# })



===== FINAL MODEL COMPARISON =====
Logistic Regression F1: Not available
Naive Bayes F1: Not available
CNN F1: Not available
HuBERT F1: Not available


In [ ]:
#final model

In [3]:
# import os
# import random
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from tqdm import tqdm
# import warnings
# import time
# warnings.filterwarnings("ignore")

# import torch
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from torch.cuda.amp import autocast, GradScaler

# import torchaudio
# from torchaudio import transforms as T

# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import f1_score

# from transformers import ASTForAudioClassification, ASTFeatureExtractor
# import wandb
# import kagglehub

2026-03-30 15:58:48.548911: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774886328.917277      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774886329.012870      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774886329.823490      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774886329.823539      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774886329.823541      55 computation_placer.cc:177] computation placer alr

In [5]:



# # ==========================================================
# # MESSY MASHUP - AST CLASSIFIER
# # Dynamic Stem Mixing + Noise Injection + Time Stretch
# # Single File Solution for Kaggle with WandB Logging
# # ==========================================================


# # ===================== CONFIGURATION =====================

# from kaggle_secrets import UserSecretsClient

# # Get API key from Kaggle secrets
# user_secrets = UserSecretsClient()
# os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# # Login
# wandb.login()

# # Init run
# wandb.init(
#     project="23f2000775-t12026",
#     name="final-ast-model",
#     config={
#         "features": "MFCC + Mel + Raw Audio",
#         "models": ["LogisticRegression", "NaiveBayes", "CNN", "HuBERT", "AST"],
#         "metric": "MacroF1"
#     }
# )

# print("W&B initialized successfully")
# # Paths
# DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
# STEMS_DIR = DATA_DIR / "genres_stems"
# MASHUPS_DIR = DATA_DIR / "mashups"
# NOISE_DIR = DATA_DIR / "ESC-50-master" / "audio"

# # Audio Params
# SR = 16000
# CHUNK_DURATION = 10      # Seconds
# OVERLAP_DURATION = 2    # Seconds
# CHUNK_LEN = SR * CHUNK_DURATION
# STEP = SR * (CHUNK_DURATION - OVERLAP_DURATION)

# # Training Params
# FOLDS = 4
# EPOCHS = 7
# BATCH_SIZE = 16
# LR = 2e-5
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # Genres
# GENRES = ["blues","classical","country","disco","hiphop",
#           "jazz","metal","pop","reggae","rock"]
# GENRE2IDX = {g:i for i,g in enumerate(GENRES)}
# IDX2GENRE = {i:g for i,g in enumerate(GENRES)}

# # Model Hub
# MODEL_HUB_PATH = None
# feature_extractor = ASTFeatureExtractor.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593"
# )

# # ===================== GLOBAL CACHES (SPEED OPTIMIZATION) =====================

# STEM_CACHE = {}  # Key: (genre, stem_type, song_id) -> waveform
# NOISE_CACHE = []
# CACHE_LOADED = False

# # Organize stems by genre and stem_type for faster lookup
# STEM_INDEX = {genre: {st: [] for st in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]} for genre in GENRES}

# def preload_assets():
#     """Loads stems and noise into RAM to avoid Disk I/O bottlenecks."""
#     global STEM_CACHE, NOISE_CACHE, CACHE_LOADED, STEM_INDEX
#     if CACHE_LOADED:
#         return

#     print(">> Preloading Stems into RAM...")
#     start_time = time.time()
   
#     # Load Stems
#     for genre in GENRES:
#         genre_path = STEMS_DIR / genre
#         if not genre_path.exists():
#             print(f"   Warning: {genre_path} not found")
#             continue
       
#         song_folders = list(genre_path.glob("*"))
#         for song_folder in tqdm(song_folders, desc=f"Loading {genre}", leave=False):
#             if not song_folder.is_dir():
#                 continue
           
#             song_id = song_folder.name
#             for stem_type in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]:
#                 p = song_folder / stem_type
#                 if p.exists():
#                     try:
#                         y, sr = torchaudio.load(p)
#                         if sr != SR:
#                             y = torchaudio.functional.resample(y, sr, SR)
#                         y = y.mean(0)  # Convert to Mono
#                         STEM_CACHE[(genre, stem_type, song_id)] = y
#                         STEM_INDEX[genre][stem_type].append((song_id, y))
#                     except Exception as e:
#                         continue
   
#     # Load Noise (ESC-50)
#     print(">> Preloading Noise (ESC-50)...")
#     if NOISE_DIR.exists():
#         noise_files = list(NOISE_DIR.glob("*.wav"))[:50]  # Limit to 50 for RAM safety
#         for p in tqdm(noise_files, desc="Loading Noise", leave=False):
#             try:
#                 y, sr = torchaudio.load(p)
#                 if sr != SR:
#                     y = torchaudio.functional.resample(y, sr, SR)
#                 y = y.mean(0)
#                 NOISE_CACHE.append(y)
#             except Exception:
#                 continue
#     else:
#         print("   Warning: ESC-50 not found. Training without environmental noise.")

#     CACHE_LOADED = True
#     print(f">> Cache Ready. Stems: {len(STEM_CACHE)}, Noise Samples: {len(NOISE_CACHE)}")
#     print(f">> Loading Time: {time.time() - start_time:.2f}s")

# # ===================== AUDIO AUGMENTATION UTILS =====================

# def get_random_stem(genre, stem_type, exclude_song_id=None):
#     """
#     Retrieves a random stem of specific type and genre from Cache.
#     Returns: (waveform, song_id) or (None, None) if not found
#     """
#     candidates = STEM_INDEX.get(genre, {}).get(stem_type, [])
   
#     if not candidates:
#         return None, None
   
#     # Filter out excluded song IDs
#     if exclude_song_id:
#         filtered = [(sid, wav) for sid, wav in candidates if sid != exclude_song_id]
#         if filtered:
#             candidates = filtered
   
#     if not candidates:
#         return None, None
       
#     song_id, waveform = random.choice(candidates)
#     return waveform.clone(), song_id

# def apply_time_stretch(waveform, rate=1.0):
#     """Simulates tempo change via resampling."""
#     if rate == 1.0 or len(waveform) == 0:
#         return waveform
   
#     new_len = int(len(waveform) / rate)
#     if new_len <= 0:
#         return waveform
   
#     indices = torch.linspace(0, len(waveform) - 1, new_len).long()
#     stretched = waveform[indices]
   
#     if len(stretched) < len(waveform):
#         stretched = F.pad(stretched, (0, len(waveform) - len(stretched)))
#     else:
#         stretched = stretched[:len(waveform)]
       
#     return stretched

# def add_noise(waveform, snr_db=10):
#     """Adds random environmental noise."""
#     if len(NOISE_CACHE) == 0 or len(waveform) == 0:
#         return waveform
       
#     noise = random.choice(NOISE_CACHE)
   
#     if len(noise) < len(waveform):
#         repeats = int(np.ceil(len(waveform) / len(noise)))
#         noise = noise.repeat(repeats)[:len(waveform)]
#     else:
#         start = random.randint(0, max(0, len(noise) - len(waveform)))
#         noise = noise[start:start+len(waveform)]
       
#     sig_power = waveform.pow(2).mean()
#     noise_power = noise.pow(2).mean()
   
#     if noise_power == 0 or sig_power == 0:
#         return waveform
       
#     target_noise_power = sig_power / (10 ** (snr_db / 10))
#     scale = torch.sqrt(target_noise_power / noise_power)
   
#     noisy_waveform = waveform + (noise * scale)
   
#     if noisy_waveform.abs().max() > 0:
#         noisy_waveform = noisy_waveform / noisy_waveform.abs().max()
       
#     return noisy_waveform

# def create_augmented_mix(genre, target_length=CHUNK_LEN):
#     """
#     Dynamically creates a messy mix simulating test conditions.
#     1. Picks 4 stems (drums, bass, vocals, others) from SAME genre.
#     2. Mixes them.
#     3. Applies Time Stretch.
#     4. Applies Noise.
#     """
#     stems = []
#     stem_types = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
#     used_song_ids = set()
   
#     for st in stem_types:
#         stem_data, song_id = get_random_stem(genre, st, exclude_song_id=used_song_ids)
#         if stem_data is not None:
#             stems.append(stem_data)
#             if song_id:
#                 used_song_ids.add(song_id)
           
#     if len(stems) == 0:
#         return None
       
#     # Align lengths
#     min_len = min(len(s) for s in stems)
#     if min_len == 0:
#         return None
   
#     stems = [s[:min_len] for s in stems]
#     mix = torch.stack(stems).sum(0)
   
#     if mix.abs().max() > 0:
#         mix = mix / mix.abs().max()
       
#     # Augmentation 1: Time Stretch (Tempo)
#     stretch_rate = random.uniform(0.9, 1.1)
#     mix = apply_time_stretch(mix, stretch_rate)
   
#     # Augmentation 2: Noise (SNR between 5dB and 20dB)
#     snr = random.uniform(5, 20)
#     mix = add_noise(mix, snr_db=snr)
   
#     # Ensure target length
#     if len(mix) < target_length:
#         mix = F.pad(mix, (0, target_length - len(mix)))
#     else:
#         start = random.randint(0, max(0, len(mix) - target_length))
#         mix = mix[start:start+target_length]
       
#     return mix

# # ===================== DATASET =====================

# class MashupDataset(Dataset):
#     def __init__(self, genres, labels, mode='train'):
#         self.genres = genres
#         self.labels = labels
#         self.mode = mode
#         self.len = len(genres)
       
#     def __len__(self):
#         return self.len
   
#     def __getitem__(self, idx):
#         genre = self.genres[idx]
#         label = self.labels[idx]
       
#         # Generate Augmented Mix
#         waveform = create_augmented_mix(genre)
       
#         # Fallback if generation fails
#         if waveform is None:
#             waveform = torch.zeros(CHUNK_LEN)
           
#         # Feature Extraction
#         inputs = feature_extractor(
#             waveform.numpy(),
#             sampling_rate=SR,
#             return_tensors="pt"
#         )
       
#         return inputs["input_values"].squeeze(0), torch.tensor(label)

# # ===================== MODEL =====================

# def build_model():
#     model = ASTForAudioClassification.from_pretrained(
#         "MIT/ast-finetuned-audioset-10-10-0.4593",
#         num_labels=10,
#         ignore_mismatched_sizes=True
#     )
#     return model.to(DEVICE)

# # ===================== TRAINING PIPELINE =====================

# def train_pipeline():
#     preload_assets()
   
#     # Construct Training List (One entry per song folder to balance)
#     train_genres = []
#     train_labels = []
   
#     song_folders = {}
#     for genre in GENRES:
#         song_folders[genre] = []
#         g_path = STEMS_DIR / genre
#         if g_path.exists():
#             for f in g_path.glob("*"):
#                 if f.is_dir():
#                     song_folders[genre].append(f)
                   
#     for genre in GENRES:
#         count = len(song_folders[genre])
#         for _ in range(count):
#             train_genres.append(genre)
#             train_labels.append(GENRE2IDX[genre])
           
#     print(f"Total Training Entries: {len(train_genres)}")
   
#     if len(train_genres) == 0:
#         print("ERROR: No training data found. Check STEMS_DIR path.")
#         return
   
#     skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
#     saved_model_paths = []
#     best_global_f1 = 0
#     fold_f1_scores = []

#     for fold, (tr_idx, val_idx) in enumerate(skf.split(train_genres, train_labels)):
#         print(f"\n========== Fold {fold} ==========")
       
#         tr_genres = [train_genres[i] for i in tr_idx]
#         tr_labels = [train_labels[i] for i in tr_idx]
#         val_genres = [train_genres[i] for i in val_idx]
#         val_labels = [train_labels[i] for i in val_idx]
       
#         train_ds = MashupDataset(tr_genres, tr_labels, mode='train')
#         val_ds = MashupDataset(val_genres, val_labels, mode='val')
       
#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
#         val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
       
#         model = build_model()
#         optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
#         scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
#         scaler = GradScaler()
       
#         best_f1 = 0
       
#         for epoch in range(EPOCHS):
#             model.train()
#             total_loss = 0
           
#             # Set Seed for Training Reproducibility
#             torch.manual_seed(epoch + fold * 100)
#             np.random.seed(epoch + fold * 100)
#             random.seed(epoch + fold * 100)
           
#             pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch+1}")
#             for xb, yb in pbar:
#                 xb = xb.to(DEVICE)
#                 yb = yb.to(DEVICE)
               
#                 optimizer.zero_grad()
               
#                 with autocast():
#                     outputs = model(input_values=xb)
#                     loss = F.cross_entropy(outputs.logits, yb)
               
#                 scaler.scale(loss).backward()
#                 scaler.step(optimizer)
#                 scaler.update()
               
#                 total_loss += loss.item()
#                 pbar.set_postfix({"loss": f"{loss.item():.4f}"})
           
#             scheduler.step()
           
#             # Validation
#             model.eval()
#             # Fixed Seed for Stable Validation Metrics
#             torch.manual_seed(42)
#             np.random.seed(42)
#             random.seed(42)
           
#             preds, targets = [], []
#             val_loss = 0
           
#             with torch.no_grad():
#                 for xb, yb in val_loader:
#                     xb = xb.to(DEVICE)
#                     yb = yb.to(DEVICE)
#                     outputs = model(input_values=xb)
#                     loss = F.cross_entropy(outputs.logits, yb)
#                     val_loss += loss.item()
#                     preds.extend(outputs.logits.argmax(1).cpu().tolist())
#                     targets.extend(yb.cpu().tolist())
           
#             f1 = f1_score(targets, preds, average="macro")
#             avg_train_loss = total_loss / len(train_loader)
#             avg_val_loss = val_loss / len(val_loader)
           
#             print(f"Fold {fold} | Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {f1:.4f}")
           
#             wandb.log({
#                 "fold": fold,
#                 "epoch": epoch + 1,
#                 "train_loss": avg_train_loss,
#                 "val_loss": avg_val_loss,
#                 "val_f1_score": f1,
#                 "learning_rate": optimizer.param_groups[0]['lr']
#             })
           
#             if f1 > best_f1:
#                 best_f1 = f1
#                 path = f"model_fold{fold}.pth"
#                 torch.save(model.state_dict(), path)
#                 saved_model_paths.append(path)
#                 if f1 > best_global_f1:
#                     best_global_f1 = f1
               
#                 wandb.log({
#                     "fold": fold,
#                     "best_f1": best_f1,
#                     "best_epoch": epoch + 1
#                 })
#                 wandb.save(path)
               
#         fold_f1_scores.append(best_f1)
#         print(f"Fold {fold} Best F1: {best_f1:.4f}")
       
#         wandb.log({f"fold_{fold}_best_f1": best_f1})

#     # Log final summary
#     mean_f1 = np.mean(fold_f1_scores)
#     std_f1 = np.std(fold_f1_scores)
   
#     print(f"\n{'='*50}")
#     print(f"TRAINING COMPLETE")
#     print(f"{'='*50}")
#     print(f"Fold F1 Scores: {[f'{f:.4f}' for f in fold_f1_scores]}")
#     print(f"Mean F1: {mean_f1:.4f} ± {std_f1:.4f}")
#     print(f"Best Global F1: {best_global_f1:.4f}")
   
#     wandb.log({
#         "mean_f1_score": mean_f1,
#         "std_f1_score": std_f1,
#         "best_global_f1": best_global_f1,
#         "fold_f1_scores": fold_f1_scores
#     })
   
#     for path in saved_model_paths:
#         wandb.save(path)


# # ===================== INFERENCE PIPELINE =====================

# @torch.no_grad()
# def inference_pipeline():
#     print(">> Starting Inference Pipeline...")
#     preload_assets()
   
#     # Load Models
#     models = []
#     model_dir = "."
   
#     try:
#         model_dir = kagglehub.model_download(MODEL_HUB_PATH)
#         print(f">> Models downloaded from: {model_dir}")
#     except Exception as e:
#         print(f">> Hub Download Failed. Using local models. {e}")
       
#     for fold in range(FOLDS):
#         model = build_model()
#         path = f"{model_dir}/model_fold{fold}.pth"
#         if os.path.exists(path):
#             model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
#             model.eval()
#             models.append(model)
#             print(f">> Loaded Fold {fold}")
#         else:
#             print(f">> Warning: Model {path} not found!")
           
#     if len(models) == 0:
#         print(">> No models loaded. Exiting.")
#         return

#     # Load Test Data
#     test_df = pd.read_csv(DATA_DIR / "test.csv")
#     sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
   
#     submission_ids = sample["id"].tolist()
   
#     # Map IDs to filenames
#     if 'id' in test_df.columns:
#         id_to_fname = dict(zip(test_df['id'], test_df['filename']))
#         final_fnames = [id_to_fname.get(sid, None) for sid in submission_ids]
#     else:
#         final_fnames = test_df["filename"].tolist()

#     preds = []
   
#     for fname in tqdm(final_fnames, desc="Inference"):
#         if fname is None:
#             preds.append(GENRES[0])
#             continue
           
#         fpath = MASHUPS_DIR / Path(fname).name
       
#         if not fpath.exists():
#             preds.append(GENRES[0])
#             continue
           
#         try:
#             mix, sr = torchaudio.load(fpath)
#             if sr != SR:
#                 mix = torchaudio.functional.resample(mix, sr, SR)
#             mix = mix.mean(0)
#         except Exception:
#             preds.append(GENRES[0])
#             continue
           
#         # Chunking
#         if len(mix) < CHUNK_LEN:
#             mix = F.pad(mix, (0, CHUNK_LEN - len(mix)))
           
#         chunks = []
#         length = len(mix)
#         num_chunks = int(np.ceil((length - CHUNK_LEN) / STEP)) + 1
#         num_chunks = min(num_chunks, 10)  # Cap for speed

#         for i in range(num_chunks):
#             start = i * STEP
#             end = start + CHUNK_LEN
#             chunk = mix[start:end]
#             if len(chunk) < CHUNK_LEN:
#                 chunk = F.pad(chunk, (0, CHUNK_LEN - len(chunk)))
#             chunks.append(chunk)
           
#         # Ensemble Prediction
#         logits_sum = 0
#         total = 0
       
#         for chunk in chunks:
#             inputs = feature_extractor(
#                 chunk.numpy(),
#                 sampling_rate=SR,
#                 return_tensors="pt"
#             )
#             xb = inputs["input_values"].to(DEVICE)
           
#             for model in models:
#                 outputs = model(input_values=xb)
#                 logits_sum += outputs.logits.cpu()
#                 total += 1
               
#         if total > 0:
#             logits_avg = logits_sum / total
#             pred_idx = logits_avg.argmax(1).item()
#             preds.append(IDX2GENRE[pred_idx])
#         else:
#             preds.append(GENRES[0])
           
#     # Save Submission
#     submission = pd.DataFrame({
#         "id": submission_ids,
#         "genre": preds
#     })
   
#     submission.to_csv("/kaggle/working/submission.csv", index=False)
#     print(">> Submission saved to /kaggle/working/submission.csv")
#     print(submission.head())
   
#     genre_counts = pd.Series(preds).value_counts().to_dict()
#     wandb.log({
#         "total_predictions": len(preds),
#         "genre_distribution": genre_counts
#     })
#     wandb.log_artifact("/kaggle/working/submission.csv", name="submission", type="submission")

# # ===================== EXECUTION =====================

# IS_TRAINING_MODE = False  # Set to False for Inference only

# if __name__ == "__main__":
#     start_time = time.time()
#     if IS_TRAINING_MODE:
#         train_pipeline()
#     else:
#         inference_pipeline()
#     print(f">> Total Execution Time: {time.time() - start_time:.2f}s")
# # =======================================================

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B initialized successfully
>> Starting Inference Pipeline...
>> Preloading Stems into RAM...


KeyboardInterrupt: 

In [1]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
import time
warnings.filterwarnings("ignore")

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchaudio
from torchaudio import transforms as T

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import ASTForAudioClassification, ASTFeatureExtractor
import wandb
import kagglehub

2026-04-05 07:32:12.225304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775374332.631364      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775374332.738947      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775374333.670174      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775374333.670213      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775374333.670216      55 computation_placer.cc:177] computation placer alr

In [ ]:
# ==========================================================
# MESSY MASHUP - AST CLASSIFIER
# Dynamic Stem Mixing + Noise Injection + Time Stretch
# Single File Solution for Kaggle with WandB Logging
# ==========================================================


# ===================== CONFIGURATION =====================

from kaggle_secrets import UserSecretsClient

# Get API key from Kaggle secrets
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login()

# Init run
wandb.init(
    project="23f2000775-t12026",
    name="final-ast-model",
    config={
        "features": "MFCC + Mel + Raw Audio",
        "models": ["LogisticRegression", "NaiveBayes", "CNN", "HuBERT", "AST"],
        "metric": "MacroF1"
    }
)

print("W&B initialized successfully")
# Paths
DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
STEMS_DIR = DATA_DIR / "genres_stems"
MASHUPS_DIR = DATA_DIR / "mashups"
NOISE_DIR = DATA_DIR / "ESC-50-master" / "audio"

# Audio Params
SR = 16000
CHUNK_DURATION = 10      # Seconds
OVERLAP_DURATION = 2    # Seconds
CHUNK_LEN = SR * CHUNK_DURATION
STEP = SR * (CHUNK_DURATION - OVERLAP_DURATION)

# Training Params
FOLDS = 4
EPOCHS = 20
BATCH_SIZE = 16
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Genres
GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
GENRE2IDX = {g:i for i,g in enumerate(GENRES)}
IDX2GENRE = {i:g for i,g in enumerate(GENRES)}

# Model Hub
MODEL_HUB_PATH = None 
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# ===================== GLOBAL CACHES (SPEED OPTIMIZATION) =====================

STEM_CACHE = {}  # Key: (genre, stem_type, song_id) -> waveform
NOISE_CACHE = []
CACHE_LOADED = False

# Organize stems by genre and stem_type for faster lookup
STEM_INDEX = {genre: {st: [] for st in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]} for genre in GENRES}

def preload_assets():
    """Loads stems and noise into RAM to avoid Disk I/O bottlenecks."""
    global STEM_CACHE, NOISE_CACHE, CACHE_LOADED, STEM_INDEX
    if CACHE_LOADED:
        return

    print(">> Preloading Stems into RAM...")
    start_time = time.time()
    
    # Load Stems
    for genre in GENRES:
        genre_path = STEMS_DIR / genre
        if not genre_path.exists():
            print(f"   Warning: {genre_path} not found")
            continue
        
        song_folders = list(genre_path.glob("*"))
        for song_folder in tqdm(song_folders, desc=f"Loading {genre}", leave=False):
            if not song_folder.is_dir():
                continue
            
            song_id = song_folder.name
            for stem_type in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]:
                p = song_folder / stem_type
                if p.exists():
                    try:
                        y, sr = torchaudio.load(p)
                        if sr != SR:
                            y = torchaudio.functional.resample(y, sr, SR)
                        y = y.mean(0)  # Convert to Mono
                        STEM_CACHE[(genre, stem_type, song_id)] = y
                        STEM_INDEX[genre][stem_type].append((song_id, y))
                    except Exception as e:
                        continue
    
    # Load Noise (ESC-50)
    print(">> Preloading Noise (ESC-50)...")
    if NOISE_DIR.exists():
        noise_files = list(NOISE_DIR.glob("*.wav"))[:50]  # Limit to 50 for RAM safety
        for p in tqdm(noise_files, desc="Loading Noise", leave=False):
            try:
                y, sr = torchaudio.load(p)
                if sr != SR:
                    y = torchaudio.functional.resample(y, sr, SR)
                y = y.mean(0)
                NOISE_CACHE.append(y)
            except Exception:
                continue
    else:
        print("   Warning: ESC-50 not found. Training without environmental noise.")

    CACHE_LOADED = True
    print(f">> Cache Ready. Stems: {len(STEM_CACHE)}, Noise Samples: {len(NOISE_CACHE)}")
    print(f">> Loading Time: {time.time() - start_time:.2f}s")

# ===================== AUDIO AUGMENTATION UTILS =====================

def get_random_stem(genre, stem_type, exclude_song_id=None):
    """
    Retrieves a random stem of specific type and genre from Cache.
    Returns: (waveform, song_id) or (None, None) if not found
    """
    candidates = STEM_INDEX.get(genre, {}).get(stem_type, [])
    
    if not candidates:
        return None, None
    
    # Filter out excluded song IDs
    if exclude_song_id:
        filtered = [(sid, wav) for sid, wav in candidates if sid != exclude_song_id]
        if filtered:
            candidates = filtered
    
    if not candidates:
        return None, None
        
    song_id, waveform = random.choice(candidates)
    return waveform.clone(), song_id

def apply_time_stretch(waveform, rate=1.0):
    """Simulates tempo change via resampling."""
    if rate == 1.0 or len(waveform) == 0:
        return waveform
    
    new_len = int(len(waveform) / rate)
    if new_len <= 0:
        return waveform
    
    indices = torch.linspace(0, len(waveform) - 1, new_len).long()
    stretched = waveform[indices]
    
    if len(stretched) < len(waveform):
        stretched = F.pad(stretched, (0, len(waveform) - len(stretched)))
    else:
        stretched = stretched[:len(waveform)]
        
    return stretched

def add_noise(waveform, snr_db=10):
    """Adds random environmental noise."""
    if len(NOISE_CACHE) == 0 or len(waveform) == 0:
        return waveform
        
    noise = random.choice(NOISE_CACHE)
    
    if len(noise) < len(waveform):
        repeats = int(np.ceil(len(waveform) / len(noise)))
        noise = noise.repeat(repeats)[:len(waveform)]
    else:
        start = random.randint(0, max(0, len(noise) - len(waveform)))
        noise = noise[start:start+len(waveform)]
        
    sig_power = waveform.pow(2).mean()
    noise_power = noise.pow(2).mean()
    
    if noise_power == 0 or sig_power == 0:
        return waveform
        
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    scale = torch.sqrt(target_noise_power / noise_power)
    
    noisy_waveform = waveform + (noise * scale)
    
    if noisy_waveform.abs().max() > 0:
        noisy_waveform = noisy_waveform / noisy_waveform.abs().max()
        
    return noisy_waveform

def create_augmented_mix(genre, target_length=CHUNK_LEN):
    """
    Dynamically creates a messy mix simulating test conditions.
    1. Picks 4 stems (drums, bass, vocals, others) from SAME genre.
    2. Mixes them.
    3. Applies Time Stretch.
    4. Applies Noise.
    """
    stems = []
    stem_types = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
    used_song_ids = set()
    
    for st in stem_types:
        stem_data, song_id = get_random_stem(genre, st, exclude_song_id=used_song_ids)
        if stem_data is not None:
            stems.append(stem_data)
            if song_id:
                used_song_ids.add(song_id)
            
    if len(stems) == 0:
        return None
        
    # Align lengths
    min_len = min(len(s) for s in stems)
    if min_len == 0:
        return None
    
    stems = [s[:min_len] for s in stems]
    mix = torch.stack(stems).sum(0)
    
    if mix.abs().max() > 0:
        mix = mix / mix.abs().max()
        
    # Augmentation 1: Time Stretch (Tempo)
    stretch_rate = random.uniform(0.9, 1.1)
    mix = apply_time_stretch(mix, stretch_rate)
    
    # Augmentation 2: Noise (SNR between 5dB and 20dB)
    snr = random.uniform(5, 20)
    mix = add_noise(mix, snr_db=snr)
    
    # Ensure target length
    if len(mix) < target_length:
        mix = F.pad(mix, (0, target_length - len(mix)))
    else:
        start = random.randint(0, max(0, len(mix) - target_length))
        mix = mix[start:start+target_length]
        
    return mix

# ===================== DATASET =====================

class MashupDataset(Dataset):
    def __init__(self, genres, labels, mode='train'):
        self.genres = genres
        self.labels = labels
        self.mode = mode
        self.len = len(genres)
        
    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        genre = self.genres[idx]
        label = self.labels[idx]
        
        # Generate Augmented Mix
        waveform = create_augmented_mix(genre)
        
        # Fallback if generation fails
        if waveform is None:
            waveform = torch.zeros(CHUNK_LEN)
            
        # Feature Extraction
        inputs = feature_extractor(
            waveform.numpy(),
            sampling_rate=SR,
            return_tensors="pt"
        )
        
        return inputs["input_values"].squeeze(0), torch.tensor(label)

# ===================== MODEL =====================

def build_model():
    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=10,
        ignore_mismatched_sizes=True
    )
    return model.to(DEVICE)

# ===================== TRAINING PIPELINE =====================

def train_pipeline():
    preload_assets()
    
    # Construct Training List (One entry per song folder to balance)
    train_genres = []
    train_labels = []
    
    song_folders = {}
    for genre in GENRES:
        song_folders[genre] = []
        g_path = STEMS_DIR / genre
        if g_path.exists():
            for f in g_path.glob("*"):
                if f.is_dir():
                    song_folders[genre].append(f)
                    
    for genre in GENRES:
        count = len(song_folders[genre])
        for _ in range(count):
            train_genres.append(genre)
            train_labels.append(GENRE2IDX[genre])
            
    print(f"Total Training Entries: {len(train_genres)}")
    
    if len(train_genres) == 0:
        print("ERROR: No training data found. Check STEMS_DIR path.")
        return
    
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
    saved_model_paths = []
    best_global_f1 = 0
    fold_f1_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_genres, train_labels)):
        print(f"\n========== Fold {fold} ==========")
        
        tr_genres = [train_genres[i] for i in tr_idx]
        tr_labels = [train_labels[i] for i in tr_idx]
        val_genres = [train_genres[i] for i in val_idx]
        val_labels = [train_labels[i] for i in val_idx]
        
        train_ds = MashupDataset(tr_genres, tr_labels, mode='train')
        val_ds = MashupDataset(val_genres, val_labels, mode='val')
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

        
        model = build_model()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = GradScaler()
        
        best_f1 = 0
        
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            
            # Set Seed for Training Reproducibility
            torch.manual_seed(epoch + fold * 100)
            np.random.seed(epoch + fold * 100)
            random.seed(epoch + fold * 100)
            
            pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch+1}")
            for xb, yb in pbar:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                
                optimizer.zero_grad()
                
                with autocast():
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            scheduler.step()
            
            # Validation
            model.eval()
            # Fixed Seed for Stable Validation Metrics
            torch.manual_seed(42)
            np.random.seed(42)
            random.seed(42)
            
            preds, targets = [], []
            val_loss = 0
            
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE)
                    yb = yb.to(DEVICE)
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                    val_loss += loss.item()
                    preds.extend(outputs.logits.argmax(1).cpu().tolist())
                    targets.extend(yb.cpu().tolist())
            
            f1 = f1_score(targets, preds, average="macro")
            avg_train_loss = total_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            
            print(f"Fold {fold} | Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {f1:.4f}")
            
            wandb.log({
                "fold": fold,
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_f1_score": f1,
                "learning_rate": optimizer.param_groups[0]['lr']
            })
            
            if f1 > best_f1:
                best_f1 = f1
                path = f"model_fold{fold}.pth"
                torch.save(model.state_dict(), path)
                saved_model_paths.append(path)
                if f1 > best_global_f1:
                    best_global_f1 = f1
                
                wandb.log({
                    "fold": fold,
                    "best_f1": best_f1,
                    "best_epoch": epoch + 1
                })
                wandb.save(path)
                
        fold_f1_scores.append(best_f1)
        print(f"Fold {fold} Best F1: {best_f1:.4f}")
        
        wandb.log({f"fold_{fold}_best_f1": best_f1})

    # Log final summary
    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    
    print(f"\n{'='*50}")
    print(f"TRAINING COMPLETE")
    print(f"{'='*50}")
    print(f"Fold F1 Scores: {[f'{f:.4f}' for f in fold_f1_scores]}")
    print(f"Mean F1: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"Best Global F1: {best_global_f1:.4f}")
    
    wandb.log({
        "mean_f1_score": mean_f1,
        "std_f1_score": std_f1,
        "best_global_f1": best_global_f1,
        "fold_f1_scores": fold_f1_scores
    })
    
    for path in saved_model_paths:
        wandb.save(path)


# ===================== INFERENCE PIPELINE =====================

@torch.no_grad()
def inference_pipeline():
    print(">> Starting Inference Pipeline...")
    preload_assets()
    
    # Load Models
    models = []
    model_dir = "."
    
    try:
        model_dir = kagglehub.model_download(MODEL_HUB_PATH)
        print(f">> Models downloaded from: {model_dir}")
    except Exception as e:
        print(f">> Hub Download Failed. Using local models. {e}")
        
    for fold in range(FOLDS):
        model = build_model()
        path = f"{model_dir}/model_fold{fold}.pth"
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
            model.eval()
            models.append(model)
            print(f">> Loaded Fold {fold}")
        else:
            print(f">> Warning: Model {path} not found!")
            
    if len(models) == 0:
        print(">> No models loaded. Exiting.")
        return

    # Load Test Data
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
    
    submission_ids = sample["id"].tolist()
    
    # Map IDs to filenames
    if 'id' in test_df.columns:
        id_to_fname = dict(zip(test_df['id'], test_df['filename']))
        final_fnames = [id_to_fname.get(sid, None) for sid in submission_ids]
    else:
        final_fnames = test_df["filename"].tolist()

    preds = []
    
    for fname in tqdm(final_fnames, desc="Inference"):
        if fname is None:
            preds.append(GENRES[0])
            continue
            
        fpath = MASHUPS_DIR / Path(fname).name
        
        if not fpath.exists():
            preds.append(GENRES[0])
            continue
            
        try:
            mix, sr = torchaudio.load(fpath)
            if sr != SR:
                mix = torchaudio.functional.resample(mix, sr, SR)
            mix = mix.mean(0)
        except Exception:
            preds.append(GENRES[0])
            continue
            
        # Chunking
        if len(mix) < CHUNK_LEN:
            mix = F.pad(mix, (0, CHUNK_LEN - len(mix)))
            
        chunks = []
        length = len(mix)
        num_chunks = int(np.ceil((length - CHUNK_LEN) / STEP)) + 1
        num_chunks = min(num_chunks, 10)  # Cap for speed

        for i in range(num_chunks):
            start = i * STEP
            end = start + CHUNK_LEN
            chunk = mix[start:end]
            if len(chunk) < CHUNK_LEN:
                chunk = F.pad(chunk, (0, CHUNK_LEN - len(chunk)))
            chunks.append(chunk)
            
        # Ensemble Prediction
        logits_sum = 0
        total = 0
        
        for chunk in chunks:
            inputs = feature_extractor(
                chunk.numpy(),
                sampling_rate=SR,
                return_tensors="pt"
            )
            xb = inputs["input_values"].to(DEVICE)
            
            for model in models:
                outputs = model(input_values=xb)
                logits_sum += outputs.logits.cpu()
                total += 1
                
        if total > 0:
            logits_avg = logits_sum / total
            pred_idx = logits_avg.argmax(1).item()
            preds.append(IDX2GENRE[pred_idx])
        else:
            preds.append(GENRES[0])
            
    # Save Submission
    submission = pd.DataFrame({
        "id": submission_ids,
        "genre": preds
    })
    
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print(">> Submission saved to /kaggle/working/submission.csv")
    print(submission.head())
    
    genre_counts = pd.Series(preds).value_counts().to_dict()
    wandb.log({
        "total_predictions": len(preds),
        "genre_distribution": genre_counts
    })
    wandb.log_artifact("/kaggle/working/submission.csv", name="submission", type="submission")

# ===================== EXECUTION =====================

IS_TRAINING_MODE = True  # Set to False for Inference only

if __name__ == "__main__":
    start_time = time.time()
    if IS_TRAINING_MODE:
        train_pipeline()
    else:
        inference_pipeline()
    print(f">> Total Execution Time: {time.time() - start_time:.2f}s")
# =======================================================

wandb: Currently logged in as: ramakrishna12-2005 (ramakrishna12-2005-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B initialized successfully


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

>> Preloading Stems into RAM...


>> Preloading Noise (ESC-50)...


...

Fold 3 Epoch 14: 100%|██████████| 47/47 [00:52<00:00,  1.13s/it, loss=0.0536]


Fold 3 | Epoch 14 | Train Loss: 0.2381 | Val Loss: 0.2025 | F1: 0.9361


Fold 3 Epoch 15: 100%|██████████| 47/47 [00:53<00:00,  1.13s/it, loss=0.1401]


Fold 3 | Epoch 15 | Train Loss: 0.2306 | Val Loss: 0.1902 | F1: 0.9252


Fold 3 Epoch 16:  38%|███▊      | 18/47 [00:20<00:32,  1.12s/it, loss=0.3196]

In [2]:
DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
STEMS_DIR = DATA_DIR / "genres_stems"
MASHUPS_DIR = DATA_DIR / "mashups"
NOISE_DIR = DATA_DIR / "ESC-50-master" / "audio"

# Audio Params
SR = 16000
CHUNK_DURATION = 10      # Seconds
OVERLAP_DURATION = 2    # Seconds
CHUNK_LEN = SR * CHUNK_DURATION
STEP = SR * (CHUNK_DURATION - OVERLAP_DURATION)

# Training Params
FOLDS = 4
EPOCHS = 20
BATCH_SIZE = 16
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Genres
GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
GENRE2IDX = {g:i for i,g in enumerate(GENRES)}
IDX2GENRE = {i:g for i,g in enumerate(GENRES)}

# Model Hub
MODEL_HUB_PATH = None 
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# ===================== GLOBAL CACHES (SPEED OPTIMIZATION) =====================

STEM_CACHE = {}  # Key: (genre, stem_type, song_id) -> waveform
NOISE_CACHE = []
CACHE_LOADED = False

# Organize stems by genre and stem_type for faster lookup
STEM_INDEX = {genre: {st: [] for st in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]} for genre in GENRES}

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [3]:
STEM_INDEX
STEM_INDEX[genre][stem_type].append((song_id, y))

{'blues': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'classical': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'country': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'disco': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'hiphop': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'jazz': {'drums.wav': [], 'bass.wav': [], 'vocals.wav': [], 'others.wav': []},
 'metal': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'pop': {'drums.wav': [], 'bass.wav': [], 'vocals.wav': [], 'others.wav': []},
 'reggae': {'drums.wav': [],
  'bass.wav': [],
  'vocals.wav': [],
  'others.wav': []},
 'rock': {'drums.wav': [], 'bass.wav': [], 'vocals.wav': [], 'others.wav': []}}